In [ ]:
import re
import time

import pandas as pd
from databricks.sdk import WorkspaceClient

workspace = WorkspaceClient()
selected = dbutils.widgets.getAll()
for name, default in {
    "warehouse_id": "",
    "catalog": "main",
    "schema": "medium_cdc_20261123",
    "order_id": "42",
}.items():
    if name not in selected:
        dbutils.widgets.text(name, default)

settings = dbutils.widgets.getAll()
warehouse_id = settings["warehouse_id"].strip()
if not warehouse_id:
    candidates = [
        wh for wh in workspace.warehouses.list()
        if wh.id and getattr(wh.state, "value", wh.state) in {"RUNNING", "STARTING", "STOPPED"}
    ]
    if len(candidates) != 1:
        raise ValueError("Set warehouse_id to a SQL warehouse you can use, then run this cell again.")
    warehouse_id = candidates[0].id

def identifier(name):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", name):
        raise ValueError("Catalog and schema must be simple SQL identifiers.")
    return f"`{name}`"

history_table = f"{identifier(settings['catalog'])}.{identifier(settings['schema'])}.lb_orders_history"
order_id = int(settings["order_id"])
if not -(2**63) <= order_id < 2**63:
    raise ValueError("order_id must fit in a signed BIGINT.")

In [ ]:
route = "/api/2.0/sql/statements"
sql = f"""SELECT order_id, amount_cents, _pg_change_type
          FROM {history_table}
          WHERE order_id = :order_id
          ORDER BY _sort_by"""
response = workspace.api_client.do("POST", route, body={
    "warehouse_id": warehouse_id,
    "statement": sql,
    "parameters": [{"name": "order_id", "type": "BIGINT", "value": str(order_id)}],
    "wait_timeout": "10s",
    "on_wait_timeout": "CONTINUE",
    "disposition": "INLINE",
    "format": "JSON_ARRAY",
})
statement_id = response["statement_id"]
deadline = time.monotonic() + 180
while response["status"]["state"] in {"PENDING", "RUNNING"}:
    if time.monotonic() >= deadline:
        raise TimeoutError(f"Warehouse statement {statement_id} is still running.")
    time.sleep(0.5)
    response = workspace.api_client.do("GET", f"{route}/{statement_id}")

status = response["status"]
if status["state"] != "SUCCEEDED":
    raise RuntimeError(f"{statement_id}: {status.get('error', {}).get('message', status['state'])}")

manifest = response["manifest"]
chunk = response["result"]
rows = list(chunk.get("data_array", []))
while chunk.get("next_chunk_index") is not None:
    index = int(chunk["next_chunk_index"])
    chunk = workspace.api_client.do("GET", f"{route}/{statement_id}/result/chunks/{index}")
    rows.extend(chunk.get("data_array", []))
if manifest.get("truncated") or len(rows) != manifest["total_row_count"]:
    raise ValueError("The SQL warehouse returned an incomplete result.")

columns = [c["name"] for c in manifest["schema"]["columns"]]
display(pd.DataFrame(rows, columns=columns))